# End-to-End Pipeline Demo

Runs config-driven steps: load data, build graph, run analysis, train model, score transactions, and surface top anomalies/subgraphs. Adjust the config path or sample size as needed.

In [1]:
from pathlib import Path
import yaml
import pandas as pd

from gnn import GraphBuilder, ModelRegistry, PipelineConfig
from gnn.data import load_transactions, validate_graph_columns
from gnn.analysis import DataAnalysis
from gnn.modeling import extract_embeddings, train_autoencoder, train_link_prediction
from gnn.transactions import TransactionFeatureBuilder, TransactionAnomalyScorer
from gnn.graph import extract_topk_subgraphs

config_candidates = [Path("config/example.yaml"), Path.cwd() / "config/example.yaml", Path.cwd().parent / "config/example.yaml"]
config_path = next((p for p in config_candidates if p.exists()), None)
if config_path is None:
    raise FileNotFoundError("Could not locate config/example.yaml; adjust path in the notebook.")
cfg = PipelineConfig.model_validate(yaml.safe_load(config_path.read_text()))
cfg

PipelineConfig(data=DataConfig(raw_path=WindowsPath('data/transactions.csv'), target='fraud_label', splits={'train': 0.7, 'val': 0.15, 'test': 0.15}, seed=7), graph=GraphConfig(src_column='from_account', dst_column='to_account', timestamp_column='event_time', amount_column='amount', target_column='fraud_label', metadata_label='demo-transactions', directed=True, edge_derivation=EdgeDerivationConfig(flow=FlowEdgeRuleConfig(enabled=True, fintech_account_column='fintech_acct_id', counterparty_account_column='counterparty_acct_id', timestamp_column='event_time', amount_column='amount', direction_column='direction', incoming_values=['CREDIT'], outgoing_values=['DEBIT'], reference_column='reference', time_window_minutes=120, amount_tolerance_pct=0.1), batch=BatchEdgeRuleConfig(enabled=True, batch_id_column='batch_id', counterparty_account_column='counterparty_acct_id', amount_column='amount'), similarity=SimilarityEdgeRuleConfig(enabled=True, counterparty_account_column='counterparty_acct_id'

In [2]:
# Load or create a small synthetic dataset if the configured path is missing.
try:
    df = load_transactions(cfg.data.raw_path)
    loaded_from_file = True
except FileNotFoundError:
    df = None
    loaded_from_file = False

if df is None or df.empty:
    loaded_from_file = False
    df = pd.DataFrame(
        {
            "fintech_acct_id": ["f1"] * 6,
            "counterparty_acct_id": ["a", "b", "c", "a", "b", "d"],
            "event_time": pd.date_range("2024-01-01", periods=6, freq="H"),
            "amount": [100, 105, 200, 50, 55, 500],
            "direction": ["CREDIT", "DEBIT", "DEBIT", "CREDIT", "DEBIT", "DEBIT"],
            "reference": ["x", "x", "y", "z", "z", "y"],
            "batch_id": ["b1", "b1", "b2", "b2", "b2", "b3"],
            "counterparty_name": ["ACME LTD", "Acme Limited", "Globex", "ACME LTD", "Globex", "Initech"],
            "counterparty_address": ["1 Main St", "1 Main Street", "2 High St", "1 Main St", "2 High St", "5 Side Ave"],
            "counterparty_city": ["NYC", "New York", "London", "NYC", "London", "Paris"],
            "counterparty_country": ["US", "US", "UK", "US", "UK", "FR"],
            "counterparty_bank_id": ["X1", "X1", "Y2", "X1", "Y2", "Z3"],
        }
    )

if cfg.graph.edge_derivation and cfg.graph.edge_derivation.flow.enabled:
    flow = cfg.graph.edge_derivation.flow
    required = [
        flow.fintech_account_column,
        flow.counterparty_account_column,
        flow.timestamp_column,
        flow.amount_column,
        flow.direction_column,
    ]
    missing = [c for c in required if c not in df.columns]
    if missing and loaded_from_file:
        raise ValueError(
            f"Missing columns for flow edge derivation: {missing}. Update config columns or rename your DataFrame columns to match."
        )
    elif missing:
        # If using the synthetic sample, ensure required columns exist
        pass

if cfg.graph.edge_derivation is None:
    validate_graph_columns(df, cfg.graph)
df.head()


C:\Users\TZ\AppData\Local\Temp\ipykernel_29840\798990443.py:15: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  "event_time": pd.date_range("2024-01-01", periods=6, freq="H"),


,fintech_acct_id,counterparty_acct_id,event_time,amount,direction,reference,batch_id,counterparty_name,counterparty_address,counterparty_city,counterparty_country,counterparty_bank_id
0,f1,a,2024-01-01 00:00:00,100,CREDIT,x,b1,ACME LTD,1 Main St,NYC,US,X1
1,f1,b,2024-01-01 01:00:00,105,DEBIT,x,b1,Acme Limited,1 Main Street,New York,US,X1
2,f1,c,2024-01-01 02:00:00,200,DEBIT,y,b2,Globex,2 High St,London,UK,Y2
3,f1,a,2024-01-01 03:00:00,50,CREDIT,z,b2,ACME LTD,1 Main St,NYC,US,X1
4,f1,b,2024-01-01 04:00:00,55,DEBIT,z,b2,Globex,2 High St,London,UK,Y2


In [3]:
# Build graph and run pre-model analysis.

# Drop feature definitions that reference columns missing in the loaded DataFrame.
def prune_features(cfg_features, df_columns):
    def keep(defn):
        return all(col in df_columns for col in (defn.columns or []))
    cfg_features.node.structural = [d for d in cfg_features.node.structural if keep(d)]
    cfg_features.node.temporal = [d for d in cfg_features.node.temporal if keep(d)]
    cfg_features.node.static = [d for d in cfg_features.node.static if keep(d)]
    cfg_features.node.risk = [d for d in cfg_features.node.risk if keep(d)]
    cfg_features.edge.structural = [d for d in cfg_features.edge.structural if keep(d)]
    cfg_features.edge.temporal = [d for d in cfg_features.edge.temporal if keep(d)]
    cfg_features.edge.static = [d for d in cfg_features.edge.static if keep(d)]
    cfg_features.edge.risk = [d for d in cfg_features.edge.risk if keep(d)]
    return cfg_features

cfg.features = prune_features(cfg.features, df.columns)

builder = GraphBuilder(cfg.graph, cfg.features)
artifacts = builder.build(df)
analysis = DataAnalysis(df, cfg.graph, cfg.features)
analysis_outputs = analysis.run_all(Path(cfg.analysis.output_dir), graph=artifacts.data)
artifacts.data

Data(
  x=[4, 11],
  edge_index=[2, 13],
  edge_attr=[13, 14],
  num_nodes=4,
  account_ids=[4],
  metadata={
    account_to_idx={
      a=0,
      b=1,
      c=2,
      d=3,
    },
    feature_schema={
      node=[11],
      edge=[14],
    },
    label='demo-transactions',
  }
)

In [4]:
# Train the configured model.
registry = ModelRegistry()
model = registry.build(cfg.model, in_channels=artifacts.data.x.shape[1])
if cfg.model.task.value == "link_prediction":
    train_metrics, model = train_link_prediction(
        model,
        artifacts.data,
        epochs=cfg.train.epochs,
        lr=cfg.train.lr,
        weight_decay=cfg.train.weight_decay,
        device=cfg.train.device,
    )
else:
    train_metrics, model = train_autoencoder(
        model,
        artifacts.data,
        epochs=cfg.train.epochs,
        lr=cfg.train.lr,
        weight_decay=cfg.train.weight_decay,
        device=cfg.train.device,
    )
train_metrics

C:\Users\TZ\repo\gnn\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


{'loss': 0.0, 'pos_score_mean': 1.0, 'neg_score_mean': nan, 'auc': nan}

In [5]:
# Transaction anomaly scoring using model embeddings.
embeddings_tensor = extract_embeddings(model, artifacts.data)
embeddings = {acc: embeddings_tensor[idx].detach().cpu().numpy() for idx, acc in enumerate(artifacts.accounts)}
tfb = TransactionFeatureBuilder(cfg.transaction_anomaly)
txn_feats = tfb.build(df, embeddings)
scores, _ = TransactionAnomalyScorer(
    method=cfg.transaction_anomaly.method,
    contamination=cfg.transaction_anomaly.contamination,
).fit_score(txn_feats.features)
df_scored = df.copy()
df_scored["transaction_anomaly_score"] = scores.values
top_anomalies = df_scored.nlargest(5, "transaction_anomaly_score")
top_anomalies

,fintech_acct_id,counterparty_acct_id,event_time,amount,direction,reference,batch_id,counterparty_name,counterparty_address,counterparty_city,counterparty_country,counterparty_bank_id,transaction_anomaly_score
5,f1,d,2024-01-01 05:00:00,500,DEBIT,y,b3,Initech,5 Side Ave,Paris,FR,Z3,0.016535
2,f1,c,2024-01-01 02:00:00,200,DEBIT,y,b2,Globex,2 High St,London,UK,Y2,-0.148815
0,f1,a,2024-01-01 00:00:00,100,CREDIT,x,b1,ACME LTD,1 Main St,NYC,US,X1,-0.299822
3,f1,a,2024-01-01 03:00:00,50,CREDIT,z,b2,ACME LTD,1 Main St,NYC,US,X1,-0.305439
4,f1,b,2024-01-01 04:00:00,55,DEBIT,z,b2,Globex,2 High St,London,UK,Y2,-0.329416


In [6]:
# Extract ego subgraphs around top accounts for interpretation.
account_scores = scores.groupby(df[cfg.transaction_anomaly.counterparty_account_column]).mean()
subgraphs = extract_topk_subgraphs(
    artifacts.data,
    account_scores,
    k=5,
    hops=cfg.transaction_anomaly.subgraph_hops,
    min_size=cfg.transaction_anomaly.subgraph_min_size,
)
subgraphs

[SubgraphResult(seed='d', nodes=['a', 'b', 'c', 'd'], score=0.016535033233459395, edge_count=13, node_count=4, edge_type_proportions={'type_flow': nan, 'type_batch': nan, 'type_similarity': 1.0}),
 SubgraphResult(seed='c', nodes=['a', 'b', 'c', 'd'], score=-0.14881529910113434, edge_count=13, node_count=4, edge_type_proportions={'type_flow': nan, 'type_batch': nan, 'type_similarity': 1.0}),
 SubgraphResult(seed='a', nodes=['a', 'b', 'c', 'd'], score=-0.302630323979013, edge_count=13, node_count=4, edge_type_proportions={'type_flow': nan, 'type_batch': nan, 'type_similarity': 1.0}),
 SubgraphResult(seed='b', nodes=['a', 'b', 'c', 'd'], score=-0.3304824844724936, edge_count=13, node_count=4, edge_type_proportions={'type_flow': nan, 'type_batch': nan, 'type_similarity': 1.0})]

In [8]:
# Transaction anomaly scoring using model embeddings.
embeddings_tensor = extract_embeddings(model, artifacts.data)
embeddings = {acc: embeddings_tensor[idx].detach().cpu().numpy() for idx, acc in enumerate(artifacts.accounts)}
tfb = TransactionFeatureBuilder(cfg.transaction_anomaly)
txn_feats = tfb.build(df, embeddings)
scorer = TransactionAnomalyScorer(
  method=cfg.transaction_anomaly.method,
  contamination=cfg.transaction_anomaly.contamination,
)
scores, scorer_model = scorer.fit_score(txn_feats.features)
df_scored = df.copy()
df_scored["transaction_anomaly_score"] = scores.values
top_anomalies = df_scored.nlargest(5, "transaction_anomaly_score")

In [9]:
# Distributions and top-N bar
_ = plot_anomaly_distributions(
  df_scored,
  score_col="transaction_anomaly_score",
  counterparty_col=cfg.transaction_anomaly.counterparty_account_column,
  top_n=10,
)

# Optional SHAP (requires `pip install shap`)
try:
  shap_values = shap_bar_for_top(
      scorer_model,
      txn_feats.features,
      top_anomalies.index,
      max_display=10,
  )
except ImportError:
  print("Install shap to view feature attributions")
except Exception as e:
  print(f"SHAP computation failed: {e}")

NameError: name 'plot_anomaly_distributions' is not defined